In [ ]:
import sys

print(sys.executable)

# Data loading and overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss

In [ ]:
df = pd.read_csv('../data/raw/avis_40k.csv', sep=';', header=0)
print("data loading and overview")

display(df.head())
display(df.info())
display(df.describe())

# Problem Definition

* As a starting point, it is always a good idea to clearly define **what the objective is**. In this project, the aim is to automatically categorize customer reviews. We are therefore dealing with a **semantic clustering** problem. Consequently, there is no target variable in the same sense as in supervised learning. However, particular attention should be paid to the metrics used to evaluate the quality of the resulting clusters.

* The feature used for semantic embedding and clustering is the `commentaire` column, which contains all customer reviews. These reviews (approximately 40,000) were scraped from the Trustpilot website and concern the company Oscaro. The other features will only be used for exploratory data analysis.

# Missing Value Handling and Feature Removal

In [ ]:
print('Missing value handling')

print(df['commentaire'].isna().sum())
print('Na % among reviews:', df['commentaire'].isna().sum()/len(df['commentaire'])*100)
print(df[df['commentaire'].isna()].head())


plt.figure(figsize=(10,10))
plt.pie(df["commentaire"].isna().value_counts(normalize=True),
            labels=['reviews', 'NaN'],
            explode=[0.1, 0.1],
            autopct="%0.2f%%",
            colors=["palevioletred", "grey"])
plt.title('NaN proportion among clients reviews')

In [ ]:
df['commentaire'] = df['commentaire'].fillna(df['titre']) # Replace NaN values with the corresponding review titles
df[['commentaire', 'auteur', 'date_avis']] = df[['commentaire', 'auteur', 'date_avis']].drop_duplicates() # duplicates deletion
df = df.drop('date_avis', axis=1) 
df = df.drop('texteServiceReply', axis=1)
df = df.drop('url', axis = 1)
df = df.drop('dateServiceReply', axis = 1)

display(df.info())
display(df.head())

# Language handling

In [ ]:
from langdetect import detect
from langdetect import DetectorFactory
DetectorFactory.seed = 34

def to_detect(text_i): # language detection
    try :
        return detect(text_i)
    except:
        return ('NA')

df["lang"] = df["commentaire"].apply(lambda x: to_detect(x))
print(df['lang'].value_counts())
print(df['lang'].describe(include='O'))

In [ ]:
display(df.info())
display(df.head())

In [ ]:
df = df[df["lang"] == "fr"] # Keep only French-language reviews
df = df.dropna(subset=['commentaire'])
df.head()
df.info()

# Temporal data visualization

In [ ]:
# chronological sorting of reviews
df["date_to_sort"] = pd.to_datetime(
    df["date_experience"],
    format="%B %d, %Y",
)

df = df.sort_values("date_to_sort").reset_index(drop=True)

# Data Splitting for Different MLOps Stages
# Initial training : 60 %
df_train = df.iloc[:24000].copy()

# Monitoring reference : 10 %
df_reference = df.iloc[24000:28000].copy()

# Simulated production : 20 %
df_production = df.iloc[28000:36000].copy()

# Final evaluation : 10 %
df_test = df.iloc[36000:].copy()


In [ ]:
month_to_number = {'January' : 1, 'February' : 2, 'March' : 3, 'April' : 4, 'May' : 5, 'June' : 6, 'July' : 7, 'August' : 8, 'September' : 9, 'October' : 10, 'November' : 11, 'December' : 12}

df['date_experience_year'] = df['date_experience'].str[-4:] 
df['date_experience_month'] = df['date_experience'].replace(month_to_number, regex=True).astype(int)
df['date_experience_year'] = df['date_experience_year'].astype(int)
df = df.drop('date_experience', axis=1)

df['comm_reply'] = np.where(df['dataServiceReply'].isna() == False, df['commentaire'] + df['dataServiceReply'], df['commentaire'])

df['length_comm'] = df['commentaire'].apply(lambda x: len(x))

plt.figure(figsize=(20,10))

plt.subplot(121)

plt.pie(df["date_experience_year"].value_counts(normalize=True),
            labels=['2025', '2026', '2024'],
            explode=[0.1, 0.1, 0.1],
            autopct="%0.2f%%",
            colors=["palevioletred", "darksalmon", "grey"])
plt.title('Distribution of reviews by year')

plt.subplot(122)

plt.pie(df["date_experience_month"].value_counts(normalize=True),
            labels=['mai', 'novembre', 'aout', 'juin', 'octobre', 'juillet', 'décembre', 'janvier', 'septembre', 'avril', 'février', 'mars'],
            #explode=[0.1, 0.1, 0.1],
            autopct="%0.2f%%")
            #colors=["palevioletred", "darksalmon", "grey"])
plt.title('Distribution of reviews by month')


df["date_experience_month"].value_counts(normalize=True)

# Distribution of ratings and review lengths

In [ ]:
plt.figure(figsize=(20,10))

plt.subplot(131)

sns.histplot(df['note'])
plt.title('Distribution of ratings')

plt.subplot(132)

sns.histplot(df['length_comm'])
plt.title('Distribution of reviews length')

plt.subplot(133)

sns.barplot(data= df[['note','length_comm']].groupby('note').median(), x='note', y='length_comm')
plt.title('review length median as a function of ratings')

In [ ]:
plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
sns.scatterplot(x=df['length_comm'], y=df['note'])
plt.title('Distribution of ratings as a function of review length')
plt.subplot(122)
ss.probplot(df['length_comm'], plot=plt)
plt.title('Review length QQ-plot')
plt.show()



#plt.subplot(223)
#sns.lmplot(x='length_comm', y='note', data=df, )

print(df[['note', 'length_comm']].corr())

print('data normality, shapiro\'s p.value:', ss.shapiro(df['length_comm'].head(5000))[1])
print('pearson correlation coef:', ss.pearsonr(df['length_comm'], df['note'])[0])
print('pearson correlation p.value:', ss.pearsonr(df['length_comm'], df['note'])[1])
print('kendall correlation coef:', ss.kendalltau(df['length_comm'], df['note'])[0])
print('kendall correlation p.value:', ss.kendalltau(df['length_comm'], df['note'])[1])

# Data export

In [ ]:
df_train.to_csv('../data/processed/avis_24k_train.csv') # initial training
df_reference.to_csv('../data/processed/avis_4k_reference.csv') # monitoring reference
df_production.to_csv('../data/processed/avis_8k_production.csv') # simulated production
df_test.to_csv('../data/processed/avis_3k_test.csv') # final test

# Wordcloud

In [ ]:
import spacy
from collections import Counter
from wordcloud import WordCloud

# loading french linguistic model
nlp = spacy.load("fr_core_news_sm")

# additional stopwords
custom_stopwords = {
    "être", "avoir", "faire", "dire",
    "aller", "voir", "pouvoir"
}

def preprocess_text(text):
    """
    Text Cleaning, lemmatization and tokenization
    """
    doc = nlp(text.lower())

    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and not token.like_url
        and not token.like_email
        and token.is_alpha
        and len(token.lemma_) > 2
        and token.lemma_.lower() not in custom_stopwords
    ]

    return tokens


texts = df[df['note']==1]['commentaire']
tokens = []
for text in texts:
    tokens.extend(preprocess_text(text))

frequencies = Counter(tokens)

# WordCloud generation
wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    max_words=100,
    collocations=False,
    random_state=42
).generate_from_frequencies(frequencies)

# Visualization
plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of the french corpus", fontsize=16)
plt.show()